# Solving the General Wave Equation: Numerical Methods vs. Neural Networks

This notebook solves the **1-D general wave equation** using two approaches:

1. **Numerical Methods** — Explicit Central Difference and Crank–Nicolson-style Implicit Scheme
2. **Neural Network** — Physics-Informed Neural Network (PINN) via PyTorch

---

## Problem Statement

The general wave equation combines wave propagation, damping, and a restoring (mass) term into a single second-order linear hyperbolic PDE:

$$\frac{\partial^2 u}{\partial t^2} + 2\alpha\,\frac{\partial u}{\partial t} + \beta\,u = c^2\,\frac{\partial^2 u}{\partial x^2}, \qquad x \in [0, L],\; t \in [0, T]$$

This is the most general constant-coefficient, second-order linear hyperbolic PDE in 1-D. It **unifies** the previous equations as special cases:

| Setting | Equation recovered |
|---------|--------------------|
| $\alpha = 0,\; \beta = 0$ | **Wave equation** |
| $\alpha = 0,\; \beta = m^2 > 0$ | **Klein–Gordon equation** |
| $\alpha > 0,\; \beta = 0$ | **Telegraph equation** |
| $\alpha > 0,\; \beta > 0$ | **General (damped Klein–Gordon)** |

The damping $2\alpha\,u_t$ dissipates energy, while the restoring term $\beta\,u$ shifts the natural frequency upward.

**Initial conditions** — first mode, released from rest:

$$u(x, 0) = \sin\!\left(\frac{\pi x}{L}\right), \qquad \frac{\partial u}{\partial t}(x, 0) = 0$$

**Boundary conditions** — fixed ends (Dirichlet):

$$u(0, t) = u(L, t) = 0$$

**Dispersion and damping** — substituting $u = e^{\lambda t}\sin(k_n x)$ yields the characteristic equation:

$$\lambda^2 + 2\alpha\,\lambda + \Omega_n^2 = 0, \qquad \Omega_n^2 = c^2 k_n^2 + \beta$$

For the first mode $k_1 = \pi/L$, with $\Omega_1 = \sqrt{c^2(\pi/L)^2 + \beta}$.

**Under-damped regime** ($\alpha < \Omega_1$) — the roots are $\lambda = -\alpha \pm i\omega_d$ with:

$$\omega_d = \sqrt{\Omega_1^2 - \alpha^2} = \sqrt{c^2\!\left(\frac{\pi}{L}\right)^{\!2} + \beta - \alpha^2}$$

**Exact solution** (with $u_t(x,0) = 0$):

$$u^*(x, t) = e^{-\alpha t}\sin\!\left(\frac{\pi x}{L}\right)\!\left[\cos(\omega_d\,t) + \frac{\alpha}{\omega_d}\sin(\omega_d\,t)\right]$$

The solution is a **damped oscillation**: the envelope decays as $e^{-\alpha t}$ while the string oscillates at frequency $\omega_d$, which is *higher* than the pure wave frequency $c\pi/L$ (due to $\beta$) but *lower* than the undamped $\Omega_1$ (due to $\alpha$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import solve_banded
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
CMAP = "inferno"

# ---- Problem parameters ---------------------
C       = 1.0       # wave speed
ALPHA   = 0.3       # damping coefficient
BETA    = 2.0       # restoring / mass parameter
L_DOM   = 1.0       # domain length [0, L]
T_END   = 3.0       # final time

# Derived quantities
K1      = np.pi / L_DOM                                # first-mode wavenumber
OMEGA1  = np.sqrt(C**2 * K1**2 + BETA)                 # undamped frequency
OMEGA_D = np.sqrt(OMEGA1**2 - ALPHA**2)                 # damped frequency
T_PER   = 2 * np.pi / OMEGA_D                          # damped oscillation period

In [ ]:
def u_init(x):
    """Initial displacement: first mode sin(πx/L)."""
    return np.sin(K1 * x)


def v_init(x):
    """Initial velocity: released from rest."""
    return np.zeros_like(x)


def u_exact(x, t):
    """Exact solution: damped oscillating first mode with frequency shift."""
    return (np.exp(-ALPHA * t) * np.sin(K1 * x)
            * (np.cos(OMEGA_D * t) + (ALPHA / OMEGA_D) * np.sin(OMEGA_D * t)))


# Quick preview — exact solution at several times
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
x_plot = np.linspace(0, L_DOM, 500)
for ax, t_snap in zip(axes, [0.0, 1.0, 2.0, 3.0]):
    ax.plot(x_plot, u_exact(x_plot, t_snap), "k-", lw=2)
    ax.fill_between(x_plot, u_exact(x_plot, t_snap), alpha=0.25, color="steelblue")
    ax.set_title(f"$t = {t_snap:.2f}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

plt.suptitle("Exact Solution — General Wave Equation (Damped + Dispersive)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"c = {C},  α = {ALPHA},  β = {BETA},  L = {L_DOM}")
print(f"Ω₁ = √(c²π²/L² + β) = {OMEGA1:.4f}  (undamped natural frequency)")
print(f"ω_d = √(Ω₁² − α²) = {OMEGA_D:.4f}  (damped frequency)")
print(f"Pure wave freq cπ/L = {C * K1:.4f}  →  β shifts up by {OMEGA1 / (C * K1):.2f}×,  α damps to ω_d")
print(f"Damped period = {T_PER:.4f},  T_end = {T_END} ({T_END / T_PER:.2f} periods)")
print(f"Amplitude at t={T_END}: e^{{-αT}} = {np.exp(-ALPHA * T_END):.4f}")

---

## Part 1 — Numerical Methods

### 1-A. Explicit Central Difference

Using central differences in both space and time on $u_{tt} + 2\alpha\,u_t + \beta\,u = c^2\,u_{xx}$, with a symmetric treatment of $u_t$:

$$\frac{u_j^{n+1} - 2u_j^n + u_j^{n-1}}{\Delta t^2} + 2\alpha\,\frac{u_j^{n+1} - u_j^{n-1}}{2\,\Delta t} + \beta\,u_j^n = c^2\,\frac{u_{j+1}^n - 2u_j^n + u_{j-1}^n}{\Delta x^2}$$

Solving for $u_j^{n+1}$:

$$u_j^{n+1} = \frac{1}{1 + \alpha\Delta t}\left[(2 - \beta\,\Delta t^2)\,u_j^n - (1 - \alpha\Delta t)\,u_j^{n-1} + r^2\!\left(u_{j+1}^n - 2u_j^n + u_{j-1}^n\right)\right]$$

where $r = c\,\Delta t/\Delta x$.

**CFL stability**: $r \leq 1$. The damping and restoring terms do not loosen the spatial CFL constraint.

**Bootstrap** (with $u_t(x,0) = 0$): Taylor expansion gives

$$u_j^1 = u_j^0 + \frac{\Delta t^2}{2}\left[c^2\,\frac{u_{j+1}^0 - 2u_j^0 + u_{j-1}^0}{\Delta x^2} - \beta\,u_j^0\right]$$

In [ ]:
def solve_explicit(Nx=200, Nt=600, T=T_END):
    """
    Explicit central-difference scheme for the general wave equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    Nt : int   Number of time steps.

    Returns
    -------
    x      : ndarray (Nx+2,)   Grid including boundary points.
    u_hist : list of ndarray   Solution snapshots.
    t_hist : list of float     Snapshot times.
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    r   = C * dt / dx

    assert r <= 1.0, f"CFL violated: r = {r:.4f} > 1"
    print(f"Explicit — dx={dx:.5f}, dt={dt:.6f}, Courant r={r:.4f}, "
          f"αΔt={ALPHA * dt:.4e}, βΔt²={BETA * dt**2:.4e}")

    coeff_fwd = 1.0 / (1.0 + ALPHA * dt)
    coeff_old = 1.0 - ALPHA * dt

    # Level n=0
    u_prev      = u_init(x).copy()
    u_prev[0]   = u_prev[-1] = 0.0

    # Level n=1 via Taylor expansion (v_init = 0)
    d2u0         = (u_prev[2:] - 2 * u_prev[1:-1] + u_prev[:-2]) / dx**2
    u_curr       = u_prev.copy()
    u_curr[1:-1] = u_prev[1:-1] + 0.5 * dt**2 * (C**2 * d2u0 - BETA * u_prev[1:-1])
    u_curr[0] = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # Explicit update with damping + restoring
        d2u          = u_curr[2:] - 2 * u_curr[1:-1] + u_curr[:-2]
        u_next       = np.zeros_like(u_curr)
        u_next[1:-1] = coeff_fwd * ((2 - BETA * dt**2) * u_curr[1:-1]
                                    - coeff_old * u_prev[1:-1]
                                    + r**2 * d2u)
        u_next[0] = u_next[-1] = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


Nx_ex, Nt_ex                    = 200, 600
x_ex, u_ex_hist, t_ex_hist      = solve_explicit(Nx_ex, Nt_ex)

# Error at final time
u_ref_ex   = u_exact(x_ex, T_END)
err_ex     = np.abs(u_ex_hist[-1] - u_ref_ex)
print(f"Max absolute error  : {err_ex.max():.3e}")
print(f"Mean absolute error : {err_ex.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_ex_hist, t_ex_hist)):
    u_ref = u_exact(x_ex, t_snap)

    axes[0, col].plot(x_ex, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_ex, u_snap, "b--", lw=1.5, label="Explicit")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_ex, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Explicit Central Difference — General Wave Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1-B. Implicit (Crank–Nicolson-style) Scheme

We write the general wave equation as $u_{tt} + 2\alpha\,u_t = \mathcal{L}u$ where $\mathcal{L}u = c^2 u_{xx} - \beta\,u$. The implicit three-level scheme:

$$\frac{u^{n+1} - 2u^n + u^{n-1}}{\Delta t^2} + 2\alpha\,\frac{u^{n+1} - u^{n-1}}{2\Delta t} = \frac{1}{4}\left(\mathcal{L}u^{n+1} + 2\,\mathcal{L}u^n + \mathcal{L}u^{n-1}\right)$$

With $s = c^2\Delta t^2/(4\Delta x^2)$ and $\mu = \beta\,\Delta t^2/4$:

$$\left[(1 + \alpha\Delta t + \mu)\,I + 2s\,(-D^2_{\text{raw}}/1)\right]\mathbf{u}^{n+1} = \text{RHS}$$

The LHS is tridiagonal with constant coefficients:

| Band | LHS coefficient |
|------|----------------|
| sub/super | $-s$ |
| diagonal | $1 + \alpha\Delta t + 2s + \mu$ |

RHS (interior points):

$$\text{RHS} = 2\mathbf{u}^n - (1 - \alpha\Delta t + \mu)\,\mathbf{u}^{n-1} + \frac{\Delta t^2}{4}\bigl(\mathcal{L}_h(2\mathbf{u}^n + \mathbf{u}^{n-1})\bigr)$$

where $\mathcal{L}_h$ is the discrete spatial operator. This scheme is **unconditionally stable** and **second-order accurate**: $\mathcal{O}(\Delta t^2, \Delta x^2)$.

In [ ]:
def solve_implicit(Nx=200, Nt=300, T=T_END):
    """
    Implicit (Crank–Nicolson-style) scheme for the general wave equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    Nt : int   Number of time steps.

    Returns
    -------
    x      : ndarray (Nx+2,)
    u_hist : list of ndarray
    t_hist : list of float
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    N   = Nx

    s   = C**2 * dt**2 / (4 * dx**2)
    mu  = BETA * dt**2 / 4
    print(f"Implicit — dx={dx:.5f}, dt={dt:.5f}, s={s:.4f}, μ={mu:.4f}, αΔt={ALPHA * dt:.4f}")

    # LHS banded matrix
    A_band          = np.zeros((3, N))
    A_band[0, 1:]   = -s                                      # super-diagonal
    A_band[1, :]    = 1 + ALPHA * dt + 2 * s + mu             # diagonal
    A_band[2, :-1]  = -s                                      # sub-diagonal

    def L_op(u_full):
        """Discrete spatial operator: c²·D²u − β·u on interior points."""
        d2u = (u_full[2:] - 2 * u_full[1:-1] + u_full[:-2]) / dx**2
        return C**2 * d2u - BETA * u_full[1:-1]

    def D2_raw(u_full):
        """Second spatial difference (without 1/dx²)."""
        return u_full[2:] - 2 * u_full[1:-1] + u_full[:-2]

    # Initialise: u^0 and bootstrap u^1
    u_prev      = u_init(x).copy()
    u_prev[0]   = u_prev[-1] = 0.0

    u_curr       = u_prev.copy()
    u_curr[1:-1] = u_prev[1:-1] + 0.5 * dt**2 * L_op(u_prev)
    u_curr[0] = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # RHS assembly
        rhs = (2 * u_curr[1:-1]
               - (1 - ALPHA * dt + mu) * u_prev[1:-1]
               + s * (2 * D2_raw(u_curr) + D2_raw(u_prev))
               - mu * (2 * u_curr[1:-1] + u_prev[1:-1]))
        # Note: the −mu·(2u^n + u^{n-1}) on RHS comes from the β term
        # in L applied to u^n and u^{n-1}, combined with the mu·u^{n-1} in the
        # coefficient (1 − αΔt + μ). Let's re-derive cleanly:
        #   RHS = 2u^n − (1−αΔt)u^{n-1} + (Δt²/4)(L_h(2u^n + u^{n-1}))
        # where L_h includes −β. Expanding:
        rhs = (2 * u_curr[1:-1]
               - (1 - ALPHA * dt) * u_prev[1:-1]
               + s * (2 * D2_raw(u_curr) + D2_raw(u_prev))
               - mu * (2 * u_curr[1:-1] + u_prev[1:-1]))

        u_next       = np.zeros_like(u_curr)
        u_next[1:-1] = solve_banded((1, 1), A_band, rhs)
        u_next[0]    = 0.0
        u_next[-1]   = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


Nx_im, Nt_im = 200, 300
x_im, u_im_hist, t_im_hist = solve_implicit(Nx_im, Nt_im)

u_ref_im = u_exact(x_im, T_END)
err_im   = np.abs(u_im_hist[-1] - u_ref_im)
print(f"Max absolute error  : {err_im.max():.3e}")
print(f"Mean absolute error : {err_im.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_im_hist, t_im_hist)):
    u_ref = u_exact(x_im, t_snap)

    axes[0, col].plot(x_im, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_im, u_snap, "g--", lw=1.5, label="Implicit")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_im, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Implicit (CN-style) — General Wave Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Grid-refinement convergence study ----------------------------------------
Nx_list = [25, 50, 100, 200, 400]
err_ex_conv, err_im_conv = [], []

for Nx in Nx_list:
    # Explicit: ensure CFL ≤ 1
    dx_c = L_DOM / (Nx + 1)
    Nt_c = max(int(np.ceil(T_END * C / (0.9 * dx_c))), 300)
    x_c, u_c, t_c   = solve_explicit(Nx, Nt_c)
    u_ex_c           = u_exact(x_c, t_c[-1])
    err_ex_conv.append(np.max(np.abs(u_c[-1] - u_ex_c)))

    # Implicit — fixed moderate Nt
    x_c2, u_c2, t_c2 = solve_implicit(Nx, Nt=600)
    u_ex_c2           = u_exact(x_c2, t_c2[-1])
    err_im_conv.append(np.max(np.abs(u_c2[-1] - u_ex_c2)))

dh = [L_DOM / (Nx + 1) for Nx in Nx_list]

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(dh, err_ex_conv, "bs-", lw=1.5, label="Explicit")
ax.loglog(dh, err_im_conv, "ro-", lw=1.5, label="Implicit (CN-style)")
ax.loglog(dh, [h**2 * err_ex_conv[0] / dh[0]**2 for h in dh],
          "b--", lw=1.0, label="$\\mathcal{O}(h^2)$")
ax.loglog(dh, [h**2 * err_im_conv[0] / dh[0]**2 for h in dh],
          "r--", lw=1.0, label="$\\mathcal{O}(h^2)$ (ref)")
ax.set_xlabel("Grid spacing $h$")
ax.set_ylabel("Max absolute error at $t = T$")
ax.set_title("Convergence Study — FDM Schemes")
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 2 — Neural Network: Physics-Informed Neural Network (PINN)

### How It Works

A PINN learns $\hat{u}_\theta(x, t)$ by minimising:

$$\mathcal{L} = \underbrace{\mathcal{L}_{IC}}_{\text{initial conditions}} + \underbrace{\mathcal{L}_{BC}}_{\text{boundary}} + \underbrace{\mathcal{L}_{PDE}}_{\text{physics residual}}$$

$$\mathcal{L}_{IC} = \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}(x_k, 0) - u_0(x_k)\Bigr)^2 + \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}_t(x_k, 0) - 0\Bigr)^2$$

$$\mathcal{L}_{BC} = \frac{1}{N_{BC}}\sum_k \left[\hat{u}(0, t_k)^2 + \hat{u}(L, t_k)^2\right]$$

$$\mathcal{L}_{PDE} = \frac{1}{N_f}\sum_k \left(\frac{\partial^2\hat{u}}{\partial t^2} + 2\alpha\,\frac{\partial\hat{u}}{\partial t} + \beta\,\hat{u} - c^2\,\frac{\partial^2\hat{u}}{\partial x^2}\right)^2$$

The PDE residual now includes **all three terms** — $u_{tt}$ (second derivative), $2\alpha\,u_t$ (first derivative, free from computing $u_{tt}$), and $\beta\,u$ (algebraic, no extra cost). Autograd handles the full general operator seamlessly.

Training: **Adam** warm-up → **L-BFGS** fine-tuning.

In [ ]:
# -----------------------------------------------------------------
# Network Architecture
# -----------------------------------------------------------------
class GeneralWavePINN(nn.Module):
    """Fully-connected PINN: (x, t) → u.
    Tanh activations ensure smooth second-order derivatives via autograd.
    """

    def __init__(self, hidden_layers=5, hidden_dim=80):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


def grad1(u, v):
    return torch.autograd.grad(
        u, v, grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]


# -----------------------------------------------------------------
# Collocation / IC / BC Points
# -----------------------------------------------------------------
N_IC    = 2000    # IC points (displacement + velocity)
N_BC    = 1000    # boundary points (each side)
N_INT   = 10000   # interior PDE collocation points

# ---- IC at t = 0 ----------------------------------
x_ic = (torch.rand(N_IC, 1) * L_DOM).requires_grad_(True)
t_ic = torch.zeros(N_IC, 1, requires_grad=True)
u_ic = torch.tensor(
    u_init(x_ic.detach().numpy()), dtype=torch.float32
)

# ---- Boundary points (x = 0 and x = L) -----------
t_bc = torch.rand(N_BC, 1) * T_END
x_bc_left  = torch.zeros(N_BC, 1)
x_bc_right = torch.full((N_BC, 1), L_DOM)

# ---- Interior PDE collocation points --------------
x_int = (torch.rand(N_INT, 1) * L_DOM).requires_grad_(True)
t_int = (torch.rand(N_INT, 1) * T_END).requires_grad_(True)

mse = nn.MSELoss()

print(f"IC  points : {N_IC}")
print(f"BC  points : {2 * N_BC}")
print(f"PDE points : {N_INT}")

In [ ]:
def compute_loss(model):
    # ---- IC loss: displacement + velocity ------
    u_pred_ic   = model(x_ic, t_ic)
    loss_ic_u   = mse(u_pred_ic, u_ic)

    u_t_ic      = grad1(u_pred_ic, t_ic)
    loss_ic_v   = mse(u_t_ic, torch.zeros_like(u_t_ic))

    loss_ic     = loss_ic_u + loss_ic_v

    # ---- Boundary loss (u = 0 at x = 0, L) -----
    u_left  = model(x_bc_left,  t_bc)
    u_right = model(x_bc_right, t_bc)
    loss_bc = mse(u_left,  torch.zeros_like(u_left)) + mse(u_right, torch.zeros_like(u_right))

    # ---- PDE residual: u_tt + 2α·u_t + β·u − c²·u_xx = 0 -----
    u_pred  = model(x_int, t_int)
    u_t     = grad1(u_pred, t_int)
    u_tt    = grad1(u_t,    t_int)
    u_x     = grad1(u_pred, x_int)
    u_xx    = grad1(u_x,    x_int)
    residual = u_tt + 2 * ALPHA * u_t + BETA * u_pred - C**2 * u_xx
    loss_pde = mse(residual, torch.zeros_like(residual))

    return loss_ic + loss_bc + loss_pde, loss_ic, loss_bc, loss_pde


# ---- Phase 1: Adam --------------------------------
model = GeneralWavePINN(hidden_layers=5, hidden_dim=80)
opt_adam = optim.Adam(model.parameters(), lr=1e-3)
ADAM_EP = 5000
history = []

for ep in range(1, ADAM_EP + 1):
    opt_adam.zero_grad()
    loss, l_ic, l_bc, l_pde = compute_loss(model)
    loss.backward()
    opt_adam.step()
    history.append(loss.item())
    if ep % 1000 == 0:
        print(f"[Adam] Ep {ep:5d} | Loss {loss.item():.5f} | "
              f"IC {l_ic.item():.5f} | BC {l_bc.item():.5f} | PDE {l_pde.item():.5f}")

print("Adam phase done.")

In [ ]:
# ---- Phase 2: L-BFGS ----
opt_lbfgs = optim.LBFGS(
    model.parameters(),
    max_iter=500, tolerance_grad=1e-9, tolerance_change=1e-12,
    history_size=50, line_search_fn="strong_wolfe"
)


def closure():
    opt_lbfgs.zero_grad()
    loss, _, _, _ = compute_loss(model)
    loss.backward()
    history.append(loss.item())
    return loss


opt_lbfgs.step(closure)
final, _, _, _ = compute_loss(model)
print(f"L-BFGS done.  Final loss: {final.item():.6f}")

# Training loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(history, color="steelblue", lw=1.0)
ax.axvline(x=ADAM_EP, color="tomato", ls="--", lw=1.5, label="Adam → L-BFGS")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Loss (log scale)")
ax.set_title("PINN Training Loss — General Wave Equation")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Evaluate PINN on a dense x grid at each snapshot time ----
model.eval()
Nev  = 500
x_ev = np.linspace(0, L_DOM, Nev)


def pinn_predict(t_val):
    xt = torch.tensor(x_ev, dtype=torch.float32).unsqueeze(1)
    tt = torch.full((Nev, 1), t_val, dtype=torch.float32)
    with torch.no_grad():
        return model(xt, tt).numpy().ravel()


snap_times_pinn = [0.0, T_END / 3, 2 * T_END / 3, T_END]
U_pinn_snaps    = [pinn_predict(t) for t in snap_times_pinn]

# PINN error at final time
U_ex_pinn   = u_exact(x_ev, T_END)
err_pinn    = np.abs(U_pinn_snaps[-1] - U_ex_pinn)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (U_snap, t_snap) in enumerate(zip(U_pinn_snaps, snap_times_pinn)):
    u_ref = u_exact(x_ev, t_snap)

    axes[0, col].plot(x_ev, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_ev, U_snap, "m--", lw=1.5, label="PINN")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(U_snap - u_ref)
    axes[1, col].plot(x_ev, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Physics-Informed Neural Network — General Wave Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"PINN — Max error at t=T: {err_pinn.max():.2e}  |  Mean error: {err_pinn.mean():.2e}")

---

## Part 3 — Side-by-Side Comparison

Solutions from all three methods at $t = T$ alongside the exact solution, plus a quantitative error summary.

In [ ]:
# Interpolate FD solutions onto the common evaluation grid
from scipy.interpolate import interp1d

u_ex_ev     = interp1d(x_ex, u_ex_hist[-1], kind="linear",
                        bounds_error=False, fill_value=0.0)(x_ev)
u_im_ev     = interp1d(x_im, u_im_hist[-1], kind="linear",
                        bounds_error=False, fill_value=0.0)(x_ev)
u_pinn_ev   = U_pinn_snaps[-1]

err_ex_ev   = np.abs(u_ex_ev   - U_ex_pinn)
err_im_ev   = np.abs(u_im_ev   - U_ex_pinn)
err_pinn_ev = np.abs(u_pinn_ev - U_ex_pinn)

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

labels  = ["Explicit", "Implicit (CN)", "PINN", "Exact"]
sols    = [u_ex_ev, u_im_ev, u_pinn_ev, U_ex_pinn]
colors  = ["steelblue", "seagreen", "mediumorchid", "black"]
errs    = [err_ex_ev, err_im_ev, err_pinn_ev]
e_lbl   = ["Explicit error", "Implicit error", "PINN error"]

for col, (lbl, sol, c) in enumerate(zip(labels, sols, colors)):
    ax = fig.add_subplot(gs[0, col])
    ax.plot(x_ev, sol, color=c, lw=2)
    ax.fill_between(x_ev, sol, alpha=0.2, color=c)
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

for col, (lbl, err) in enumerate(zip(e_lbl, errs)):
    ax = fig.add_subplot(gs[1, col])
    ax.plot(x_ev, err, "r-", lw=1.5)
    ax.set_title(f"{lbl}  (max {err.max():.1e})", fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("|error|")
    ax.grid(alpha=0.3)

# Overlay plot for direct comparison
ax_ov = fig.add_subplot(gs[1, 3])
ax_ov.plot(x_ev, U_ex_pinn,  "k-",  lw=2,   label="Exact")
ax_ov.plot(x_ev, u_ex_ev,    "b--", lw=1.5, label="Explicit")
ax_ov.plot(x_ev, u_im_ev,    "g:",  lw=1.5, label="Implicit")
ax_ov.plot(x_ev, u_pinn_ev,  "m-.", lw=1.5, label="PINN")
ax_ov.set_title("$t = T$ overlay", fontsize=10)
ax_ov.set_xlabel("$x$")
ax_ov.set_ylabel("$u$")
ax_ov.legend(fontsize=8)
ax_ov.grid(alpha=0.3)

fig.suptitle(f"Final Comparison at $t = T = {T_END}$ — General Wave Equation",
             fontsize=13, fontweight="bold")
plt.show()

In [ ]:
# ---- Quantitative error summary ----
def l2_norm(f_arr, x_arr):
    return np.sqrt(np.trapezoid(f_arr**2, x_arr))


# Peak amplitude at t=T
amp_exact = np.max(np.abs(U_ex_pinn))
amp_ex    = np.max(np.abs(u_ex_ev))
amp_im    = np.max(np.abs(u_im_ev))
amp_pinn  = np.max(np.abs(u_pinn_ev))

env_T = np.exp(-ALPHA * T_END)

print("=" * 72)
print(f"{'Method':<22} {'Max error':>12} {'Mean error':>12} {'L² error':>12} {'Peak |u|':>10}")
print("-" * 72)
print(f"{'Exact':22} {'—':>12} {'—':>12} {'—':>12} {amp_exact:>10.4f}")
print(f"{'Explicit':22} {err_ex_ev.max():>12.3e} {err_ex_ev.mean():>12.3e} "
      f"{l2_norm(err_ex_ev, x_ev):>12.3e} {amp_ex:>10.4f}")
print(f"{'Implicit (CN-style)':22} {err_im_ev.max():>12.3e} {err_im_ev.mean():>12.3e} "
      f"{l2_norm(err_im_ev, x_ev):>12.3e} {amp_im:>10.4f}")
print(f"{'PINN':22} {err_pinn_ev.max():>12.3e} {err_pinn_ev.mean():>12.3e} "
      f"{l2_norm(err_pinn_ev, x_ev):>12.3e} {amp_pinn:>10.4f}")
print("=" * 72)
print(f"\nDamping envelope: e^{{-αT}} = e^{{-{ALPHA}·{T_END}}} = {env_T:.4f}")
print(f"Undamped natural frequency Ω₁ = {OMEGA1:.4f},  damped frequency ω_d = {OMEGA_D:.4f}")
print(f"The restoring term β raises the frequency while damping α decays the amplitude.")

---

## Summary

### About the General Wave Equation

The general wave equation $u_{tt} + 2\alpha\,u_t + \beta\,u = c^2\,u_{xx}$ is the **most general constant-coefficient, second-order linear hyperbolic PDE** in one spatial dimension. Key properties:

- It **unifies** the wave equation ($\alpha = \beta = 0$), Klein–Gordon ($\alpha = 0$), and telegraph ($\beta = 0$) equations as special cases — a single framework covering all the previous notebooks.
- The **dispersion relation** $\omega^2 + 2i\alpha\omega - (c^2 k^2 + \beta) = 0$ shows two competing effects: the restoring term $\beta$ *raises* the natural frequency $\Omega = \sqrt{c^2 k^2 + \beta}$, while the damping $\alpha$ *lowers* the oscillation frequency to $\omega_d = \sqrt{\Omega^2 - \alpha^2}$ and introduces exponential decay.
- **Three regimes** based on $\alpha$ vs $\Omega$:
  - **Under-damped** ($\alpha < \Omega$): decaying oscillations at frequency $\omega_d$
  - **Critically damped** ($\alpha = \Omega$): fastest non-oscillatory return to equilibrium
  - **Over-damped** ($\alpha > \Omega$): pure exponential decay, no oscillations
- **Energy**: $E(t) = \frac{1}{2}\int[u_t^2 + c^2 u_x^2 + \beta\,u^2]\,dx$ decays at rate $dE/dt = -2\alpha\int u_t^2\,dx \leq 0$.

### Method Comparison

| Aspect | Explicit (Central Diff.) | Implicit (CN-style) | PINN |
|--------|------------------------|---------------------|------|
| **Core idea** | Central differences + damping/mass corrections | Three-level implicit averaging of full operator | Minimise PDE + IC + BC residuals |
| **Accuracy** | $\mathcal{O}(\Delta t^2,\,h^2)$ | $\mathcal{O}(\Delta t^2,\,h^2)$ | Depends on training |
| **Stability** | Conditional (CFL: $r \leq 1$) | Unconditional | Unconditional |
| **Damping + mass** | Modify update coefficients | Absorbed into tridiagonal LHS and RHS | $u_t$ free from $u_{tt}$; $\beta u$ algebraic |
| **Linear system** | None (explicit) | Tridiagonal solve per step | None (gradient descent) |
| **Mesh required** | Yes — uniform 1D grid | Yes — uniform 1D grid | No — meshfree |
| **Best for** | Quick prototyping, moderate parameters | Large time steps, stiff parameters | High-dimensional or parametric studies |

### Key Observations

- The **explicit scheme** picks up the damping via coefficients $(1 + \alpha\Delta t)^{-1}$ and $(1 - \alpha\Delta t)$, plus the restoring term as $-\beta\Delta t^2\,u_j^n$. Implementation is straightforward — just two extra scalar multiplications per grid point compared to the pure wave equation.
- The **implicit scheme** absorbs *both* $\alpha$ and $\beta$ into the tridiagonal diagonal entry: $1 + \alpha\Delta t + 2s + \mu$. Both terms *improve* diagonal dominance, making the system increasingly well-conditioned as damping or mass increase.
- The **PINN** naturally handles the full operator $u_{tt} + 2\alpha u_t + \beta u - c^2 u_{xx}$ since $u_t$ is an intermediate result of computing $u_{tt}$, and $\beta u$ is purely algebraic. The general wave equation is therefore **no harder** for a PINN than the standard wave equation.